# Semana 10 — SQL Avançado

**Curso:** Análise de Dados com Python — SENAI (Turma T5)
**UC (MSEP):** Manipulação de Dados com Python e SQL (150h) — Bloco 5, Semana 10

Nas Semanas 08 e 09 você aprendeu a consultar (SELECT), filtrar (WHERE), resumir (GROUP BY) e relacionar tabelas (JOIN, VIEW). Esta semana fecha o ciclo de SQL do curso: você aprende a alterar dados de verdade (INSERT, UPDATE, DELETE), a usar funções prontas de número/texto/data, a escrever consultas dentro de consultas (subqueries), e conhece — numa primeira olhada, sem profundidade — variáveis, functions e transactions no PostgreSQL.

> Em cada tópico abaixo: **exemplos resolvidos** + **1 atividade prática** para você fazer sozinho(a).

### 🧭 De onde você vem: as Semanas 08 e 09

Você já tem `sabor_caseiro` com as tabelas `pedidos` e `produtos`, sabe escrever SELECT/WHERE/GROUP BY, e sabe relacionar tabelas com INNER JOIN e VIEW. Esta semana usa as MESMAS tabelas — e ensina, pela primeira vez, a MUDAR os dados que estão nelas (não só consultar).

---
### 🟢 Abertura — Semana 10: SQL Avançado

**O que você vai aprender nesta semana:**
- Operações CRUD completas: INSERT, UPDATE e DELETE (com WHERE, sempre)
- Funções de número, texto e data prontas do PostgreSQL
- Subconsultas (subqueries): uma consulta dentro de outra
- Uma primeira olhada em variáveis/blocos anônimos, functions e transactions

---
## 1. Operações CRUD no PostgreSQL

CRUD é a sigla para as 4 operações básicas sobre dado: **C**reate (INSERT), **R**ead (SELECT, que você já sabe), **U**pdate (UPDATE) e **D**elete (DELETE). Diferente do SELECT, os três comandos abaixo MUDAM os dados de verdade — por isso, sempre com cuidado redobrado com o WHERE.

| Comando | O que faz |
|---|---|
| `INSERT INTO tabela (col1, col2) VALUES (v1, v2)` | Adiciona uma nova linha |
| `UPDATE tabela SET col = valor WHERE ...` | Altera linhas que já existem |
| `DELETE FROM tabela WHERE ...` | Remove linhas |

> ⚠️ **Sem WHERE, UPDATE e DELETE afetam TODAS as linhas da tabela** — sem exceção, sem confirmação. Sempre confira o WHERE antes de rodar.

### 🔹 Exemplo 1 — INSERT: adicionando um novo pedido

📖 **Antes do código:** `INSERT INTO pedidos (...) VALUES (...)` adiciona 1 linha nova. `id_pedido`, sendo `SERIAL`, não precisa de valor — o banco gera sozinho.

In [ ]:
import psycopg2

try:
    conexao = psycopg2.connect(
        host="localhost",
        dbname="sabor_caseiro",
        user="postgres",
        password="SUA_SENHA_AQUI",
    )
    cursor = conexao.cursor()
    cursor.execute("""
        INSERT INTO pedidos (cliente, filial, item, valor, data_hora)
        VALUES ('Beatriz Ramos', 'Centro', 'Suco Natural', 9.50, NOW());
    """)
    conexao.commit()
    print("Executado com sucesso!")
    conexao.close()
except ModuleNotFoundError:
    print("psycopg2 ainda não foi instalado nesta sessão — reinstale com %pip install psycopg2-binary.")
except psycopg2.OperationalError as erro:
    print("Ainda não conectou. Confira, nos Serviços do Windows, se 'postgresql-x64-...' está 'Em execução', e se host/usuário/senha/banco estão corretos.")
    print(f"Erro real do Python: {erro}")


### 🔹 Exemplo 2 — UPDATE e DELETE, sempre com WHERE

📖 **Antes do código:** `UPDATE` altera o valor de uma coluna nas linhas que batem com o WHERE. `DELETE FROM` remove as linhas que batem com o WHERE — a tabela continua existindo, só perde aquelas linhas.

In [ ]:
import psycopg2

try:
    conexao = psycopg2.connect(
        host="localhost",
        dbname="sabor_caseiro",
        user="postgres",
        password="SUA_SENHA_AQUI",
    )
    cursor = conexao.cursor()
    cursor.execute("""
        UPDATE produtos
        SET preco = 10.50
        WHERE item = 'Suco Natural';
    """)
    conexao.commit()
    print("Executado com sucesso!")
    conexao.close()
except ModuleNotFoundError:
    print("psycopg2 ainda não foi instalado nesta sessão — reinstale com %pip install psycopg2-binary.")
except psycopg2.OperationalError as erro:
    print("Ainda não conectou. Confira, nos Serviços do Windows, se 'postgresql-x64-...' está 'Em execução', e se host/usuário/senha/banco estão corretos.")
    print(f"Erro real do Python: {erro}")


### ✏️ Atividade Prática 1 — Sua vez de programar

**Contextualização:** o cliente Beatriz Ramos (que você inseriu no Exemplo 1) cancelou o pedido.

**Comando:** escreva um `DELETE` que remova só o pedido da Beatriz Ramos com item `'Suco Natural'` (use `WHERE cliente = 'Beatriz Ramos' AND item = 'Suco Natural'` para não apagar outros pedidos por engano).

In [ ]:
import psycopg2

try:
    conexao = psycopg2.connect(
        host="localhost",
        dbname="sabor_caseiro",
        user="postgres",
        password="SUA_SENHA_AQUI",
    )
    cursor = conexao.cursor()
    cursor.execute("""
        DELETE FROM pedidos
        WHERE cliente = 'Beatriz Ramos' AND item = 'Suco Natural';
    """)
    conexao.commit()
    print("Executado com sucesso!")
    conexao.close()
except ModuleNotFoundError:
    print("psycopg2 ainda não foi instalado nesta sessão — reinstale com %pip install psycopg2-binary.")
except psycopg2.OperationalError as erro:
    print("Ainda não conectou. Confira, nos Serviços do Windows, se 'postgresql-x64-...' está 'Em execução', e se host/usuário/senha/banco estão corretos.")
    print(f"Erro real do Python: {erro}")


---
## 2. Funções de Número, Texto e Data

O PostgreSQL vem com dezenas de funções prontas pra transformar o resultado de uma consulta, sem precisar processar nada em Python depois.

| Categoria | Função | O que faz |
|---|---|---|
| Número | `ROUND(valor, casas)` | Arredonda com N casas decimais |
| Texto | `UPPER(texto)` / `LOWER(texto)` | Deixa maiúsculo / minúsculo |
| Texto | `CONCAT(a, b, ...)` | Junta textos |
| Texto | `LENGTH(texto)` | Conta caracteres |
| Data | `NOW()` | Data e hora atuais |
| Data | `EXTRACT(MONTH FROM data)` | Extrai uma parte da data (mês, ano, dia) |

### 🔹 Exemplo 1 — Funções de texto e número

📖 **Antes do código:** `UPPER(cliente)` deixa o nome em maiúsculas; `ROUND(valor, 0)` arredonda o valor sem casas decimais — as duas funções são aplicadas coluna a coluna, no resultado da consulta, sem alterar o dado guardado na tabela.

In [ ]:
import psycopg2

try:
    conexao = psycopg2.connect(
        host="localhost",
        dbname="sabor_caseiro",
        user="postgres",
        password="SUA_SENHA_AQUI",
    )
    cursor = conexao.cursor()
    cursor.execute("""
        SELECT UPPER(cliente) AS cliente_maiusculo,
               ROUND(valor, 0) AS valor_arredondado
        FROM pedidos
        LIMIT 5;
    """)
    colunas = [desc[0] for desc in cursor.description]
    resultado = cursor.fetchall()
    import pandas as pd
    display(pd.DataFrame(resultado, columns=colunas))
    conexao.close()
except ModuleNotFoundError:
    print("psycopg2 ainda não foi instalado nesta sessão — reinstale com %pip install psycopg2-binary.")
except psycopg2.OperationalError as erro:
    print("Ainda não conectou. Confira, nos Serviços do Windows, se 'postgresql-x64-...' está 'Em execução', e se host/usuário/senha/banco estão corretos.")
    print(f"Erro real do Python: {erro}")


### 🔹 Exemplo 2 — Funções de data

📖 **Antes do código:** `EXTRACT(MONTH FROM data_hora)` extrai só o número do mês de uma data completa. `EXTRACT(DOW FROM data_hora)` extrai o dia da semana (0 = domingo).

In [ ]:
import psycopg2

try:
    conexao = psycopg2.connect(
        host="localhost",
        dbname="sabor_caseiro",
        user="postgres",
        password="SUA_SENHA_AQUI",
    )
    cursor = conexao.cursor()
    cursor.execute("""
        SELECT cliente, data_hora,
               EXTRACT(MONTH FROM data_hora) AS mes,
               EXTRACT(DOW FROM data_hora) AS dia_semana
        FROM pedidos
        LIMIT 5;
    """)
    colunas = [desc[0] for desc in cursor.description]
    resultado = cursor.fetchall()
    import pandas as pd
    display(pd.DataFrame(resultado, columns=colunas))
    conexao.close()
except ModuleNotFoundError:
    print("psycopg2 ainda não foi instalado nesta sessão — reinstale com %pip install psycopg2-binary.")
except psycopg2.OperationalError as erro:
    print("Ainda não conectou. Confira, nos Serviços do Windows, se 'postgresql-x64-...' está 'Em execução', e se host/usuário/senha/banco estão corretos.")
    print(f"Erro real do Python: {erro}")


### ✏️ Atividade Prática 2 — Sua vez de programar

**Contextualização:** o time de marketing quer uma lista com o nome do cliente em maiúsculas e o valor do pedido arredondado, pra montar um cartaz de agradecimento aos clientes do mês.

**Comando:** escreva uma consulta com `UPPER(cliente)` e `ROUND(valor, 0)`, ordenada pelo valor arredondado em ordem decrescente, trazendo as 5 primeiras linhas.

In [ ]:
import psycopg2

try:
    conexao = psycopg2.connect(
        host="localhost",
        dbname="sabor_caseiro",
        user="postgres",
        password="SUA_SENHA_AQUI",
    )
    cursor = conexao.cursor()
    cursor.execute("""
        SELECT UPPER(cliente) AS cliente_maiusculo,
               ROUND(valor, 0) AS valor_arredondado
        FROM pedidos
        ORDER BY valor_arredondado DESC
        LIMIT 5;
    """)
    colunas = [desc[0] for desc in cursor.description]
    resultado = cursor.fetchall()
    import pandas as pd
    display(pd.DataFrame(resultado, columns=colunas))
    conexao.close()
except ModuleNotFoundError:
    print("psycopg2 ainda não foi instalado nesta sessão — reinstale com %pip install psycopg2-binary.")
except psycopg2.OperationalError as erro:
    print("Ainda não conectou. Confira, nos Serviços do Windows, se 'postgresql-x64-...' está 'Em execução', e se host/usuário/senha/banco estão corretos.")
    print(f"Erro real do Python: {erro}")


---
## 3. Subqueries

Uma **subconsulta** é uma consulta escrita DENTRO de outra consulta. A subconsulta roda primeiro, produz uma lista de valores, e essa lista é usada como filtro pela consulta principal.

| Comando | O que faz |
|---|---|
| `WHERE coluna IN (SELECT ...)` | "Está nessa lista?" |
| `WHERE coluna NOT IN (SELECT ...)` | "NÃO está nessa lista?" |

### 🔹 Exemplo 1 — Subconsulta com IN

📖 **Antes do código:** a subconsulta `(SELECT item FROM produtos WHERE categoria = 'Prato Principal')` roda primeiro e devolve uma lista de itens; a consulta de fora traz só os pedidos cujo `item` está nessa lista.

In [ ]:
import psycopg2

try:
    conexao = psycopg2.connect(
        host="localhost",
        dbname="sabor_caseiro",
        user="postgres",
        password="SUA_SENHA_AQUI",
    )
    cursor = conexao.cursor()
    cursor.execute("""
        SELECT cliente, item, valor
        FROM pedidos
        WHERE item IN (
            SELECT item FROM produtos WHERE categoria = 'Prato Principal'
        )
        ORDER BY valor DESC;
    """)
    colunas = [desc[0] for desc in cursor.description]
    resultado = cursor.fetchall()
    import pandas as pd
    display(pd.DataFrame(resultado, columns=colunas))
    conexao.close()
except ModuleNotFoundError:
    print("psycopg2 ainda não foi instalado nesta sessão — reinstale com %pip install psycopg2-binary.")
except psycopg2.OperationalError as erro:
    print("Ainda não conectou. Confira, nos Serviços do Windows, se 'postgresql-x64-...' está 'Em execução', e se host/usuário/senha/banco estão corretos.")
    print(f"Erro real do Python: {erro}")


### 🔹 Exemplo 2 — Subconsulta com NOT IN

📖 **Antes do código:** `NOT IN` inverte a lógica — traz só quem NÃO está na lista da subconsulta. Aqui, os produtos que nunca foram pedidos.

In [ ]:
import psycopg2

try:
    conexao = psycopg2.connect(
        host="localhost",
        dbname="sabor_caseiro",
        user="postgres",
        password="SUA_SENHA_AQUI",
    )
    cursor = conexao.cursor()
    cursor.execute("""
        SELECT nome_produto
        FROM (SELECT item AS nome_produto FROM produtos) AS todos_produtos
        WHERE nome_produto NOT IN (
            SELECT DISTINCT item FROM pedidos
        );
    """)
    colunas = [desc[0] for desc in cursor.description]
    resultado = cursor.fetchall()
    import pandas as pd
    display(pd.DataFrame(resultado, columns=colunas))
    conexao.close()
except ModuleNotFoundError:
    print("psycopg2 ainda não foi instalado nesta sessão — reinstale com %pip install psycopg2-binary.")
except psycopg2.OperationalError as erro:
    print("Ainda não conectou. Confira, nos Serviços do Windows, se 'postgresql-x64-...' está 'Em execução', e se host/usuário/senha/banco estão corretos.")
    print(f"Erro real do Python: {erro}")


### ✏️ Atividade Prática 3 — Sua vez de programar

**Contextualização:** a diretoria quer saber quais clientes fizeram pedidos de itens da categoria "Bebida", pra uma campanha de desconto em bebidas.

**Comando:** escreva uma consulta com `WHERE item IN (SELECT item FROM produtos WHERE categoria = 'Bebida')`, trazendo `cliente` e `item`.

In [ ]:
import psycopg2

try:
    conexao = psycopg2.connect(
        host="localhost",
        dbname="sabor_caseiro",
        user="postgres",
        password="SUA_SENHA_AQUI",
    )
    cursor = conexao.cursor()
    cursor.execute("""
        SELECT cliente, item
        FROM pedidos
        WHERE item IN (
            SELECT item FROM produtos WHERE categoria = 'Bebida'
        );
    """)
    colunas = [desc[0] for desc in cursor.description]
    resultado = cursor.fetchall()
    import pandas as pd
    display(pd.DataFrame(resultado, columns=colunas))
    conexao.close()
except ModuleNotFoundError:
    print("psycopg2 ainda não foi instalado nesta sessão — reinstale com %pip install psycopg2-binary.")
except psycopg2.OperationalError as erro:
    print("Ainda não conectou. Confira, nos Serviços do Windows, se 'postgresql-x64-...' está 'Em execução', e se host/usuário/senha/banco estão corretos.")
    print(f"Erro real do Python: {erro}")


---
## 4. Uma Primeira Olhada: Variáveis, Functions e Transactions

Esta seção é uma **introdução**, sem profundidade — o objetivo é você reconhecer esses 3 recursos e saber que existem, não dominá-los ainda.

### Variáveis e blocos anônimos

Um **bloco anônimo** (`DO $$ ... $$`) permite rodar um pedacinho de código SQL com lógica (variável, `IF`, laço), sem precisar salvar como função. `DECLARE` cria uma variável; `:=` atribui um valor a ela.

In [ ]:
try:
    import psycopg2

    conexao = psycopg2.connect(
        host="localhost", dbname="sabor_caseiro", user="postgres", password="SUA_SENHA_AQUI",
    )
    cursor = conexao.cursor()
    cursor.execute("""
        DO $$
        DECLARE
            total_pedidos INT;
        BEGIN
            SELECT COUNT(*) INTO total_pedidos FROM pedidos;
            RAISE NOTICE 'Total de pedidos: %', total_pedidos;
        END $$;
    """)
    conexao.commit()
    print("Bloco executado! (a mensagem RAISE NOTICE aparece no log do servidor, não no Python)")
    conexao.close()
except ModuleNotFoundError:
    print("psycopg2 ainda não foi instalado nesta sessão.")
except psycopg2.OperationalError as erro:
    print(f"Erro real do Python: {erro}")

### Functions

Uma **function** salva um bloco de lógica reutilizável, que pode ser chamado dentro de um SELECT — parecido com uma função Python, mas vivendo dentro do banco.

In [ ]:
try:
    import psycopg2

    conexao = psycopg2.connect(
        host="localhost", dbname="sabor_caseiro", user="postgres", password="SUA_SENHA_AQUI",
    )
    cursor = conexao.cursor()
    cursor.execute("""
        CREATE OR REPLACE FUNCTION valor_com_desconto(preco_original NUMERIC, percentual NUMERIC)
        RETURNS NUMERIC AS $$
        BEGIN
            RETURN preco_original - (preco_original * percentual / 100);
        END;
        $$ LANGUAGE plpgsql;
    """)
    conexao.commit()
    print("Function valor_com_desconto criada!")

    cursor.execute("SELECT item, preco, valor_com_desconto(preco, 10) AS com_desconto FROM produtos;")
    colunas = [desc[0] for desc in cursor.description]
    resultado = cursor.fetchall()
    import pandas as pd
    display(pd.DataFrame(resultado, columns=colunas))
    conexao.close()
except ModuleNotFoundError:
    print("psycopg2 ainda não foi instalado nesta sessão.")
except psycopg2.OperationalError as erro:
    print(f"Erro real do Python: {erro}")

### Transactions

Uma **transaction** agrupa vários comandos (INSERT/UPDATE/DELETE) numa unidade só: ou TODOS funcionam (`COMMIT`), ou NENHUM é aplicado (`ROLLBACK`) — útil quando uma operação depende de outra dar certo.

In [ ]:
try:
    import psycopg2

    conexao = psycopg2.connect(
        host="localhost", dbname="sabor_caseiro", user="postgres", password="SUA_SENHA_AQUI",
    )
    cursor = conexao.cursor()

    try:
        cursor.execute("UPDATE produtos SET preco = 33.90 WHERE item = 'Marmita Executiva';")
        cursor.execute("UPDATE produtos SET preco = 46.00 WHERE item = 'Feijoada Completa';")
        conexao.commit()
        print("Transaction concluída — as 2 alterações foram aplicadas juntas.")
    except Exception as erro_transacao:
        conexao.rollback()
        print("Algo deu errado — NENHUMA alteração foi aplicada:", erro_transacao)

    conexao.close()
except ModuleNotFoundError:
    print("psycopg2 ainda não foi instalado nesta sessão.")
except psycopg2.OperationalError as erro:
    print(f"Erro real do Python: {erro}")

### ✏️ Atividade Prática 4 — Sua vez de programar

**Contextualização:** a equipe de TI quer uma function pronta que calcule o valor de um pedido já com um acréscimo de taxa de entrega, pra usar em vários relatórios sem repetir a conta.

**Comando:** crie uma function `valor_com_taxa(valor_original NUMERIC, taxa NUMERIC) RETURNS NUMERIC` que retorne `valor_original + taxa`, e teste chamando ela num `SELECT` sobre a tabela `pedidos` (some uma taxa fixa de 5 a cada pedido).

In [ ]:
try:
    import psycopg2

    conexao = psycopg2.connect(
        host="localhost", dbname="sabor_caseiro", user="postgres", password="SUA_SENHA_AQUI",
    )
    cursor = conexao.cursor()
    cursor.execute("""
        CREATE OR REPLACE FUNCTION valor_com_taxa(valor_original NUMERIC, taxa NUMERIC)
        RETURNS NUMERIC AS $$
        BEGIN
            RETURN valor_original + taxa;
        END;
        $$ LANGUAGE plpgsql;
    """)
    conexao.commit()
    print("Function valor_com_taxa criada!")

    cursor.execute("SELECT cliente, valor, valor_com_taxa(valor, 5) AS valor_final FROM pedidos LIMIT 5;")
    colunas = [desc[0] for desc in cursor.description]
    resultado = cursor.fetchall()
    import pandas as pd
    display(pd.DataFrame(resultado, columns=colunas))
    conexao.close()
except ModuleNotFoundError:
    print("psycopg2 ainda não foi instalado nesta sessão.")
except psycopg2.OperationalError as erro:
    print(f"Erro real do Python: {erro}")

---
## 5. Treino em Squads — Sexta-feira (Encontro 3)

Cada squad recebe uma tabela pronta e precisa escrever os comandos CRUD + uma função de texto/número pedidos.

### Squad B — Academia

**Contextualização:** a academia quer registrar um novo aluno, depois corrigir o plano dele, e por fim mostrar o nome de todos em maiúsculas.

In [ ]:
try:
    import psycopg2

    conexao = psycopg2.connect(
        host="localhost", dbname="sabor_caseiro", user="postgres", password="SUA_SENHA_AQUI",
    )
    cursor = conexao.cursor()
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS squad_b_alunos (
            id_aluno SERIAL PRIMARY KEY, nome VARCHAR(100), plano VARCHAR(30)
        );
    """)
    cursor.execute("TRUNCATE TABLE squad_b_alunos RESTART IDENTITY;")
    cursor.executemany(
        "INSERT INTO squad_b_alunos (nome, plano) VALUES (%s, %s);",
        [("Fábio Souza", "Mensal"), ("Larissa Melo", "Anual")],
    )
    conexao.commit()
    print("Tabela squad_b_alunos pronta!")

    # 1) INSERT: adicione o aluno "Camila Duarte" com plano "Trimestral"
    # 2) UPDATE: corrija o plano de "Fábio Souza" para "Anual"
    # 3) SELECT com UPPER(nome): mostre todos os nomes em maiúsculas
    cursor.execute("""
        -- SUAS CONSULTAS AQUI
    """)
    print(cursor.fetchall())
    conexao.close()
except ModuleNotFoundError:
    print("psycopg2 ainda não foi instalado nesta sessão.")
except psycopg2.OperationalError as erro:
    print(f"Erro real do Python: {erro}")

### 🗣️ Debate coletivo (após as apresentações)

- Algum squad esqueceu o WHERE no UPDATE ou DELETE? O que teria acontecido?
- Alguém usou uma subconsulta pra resolver o desafio, em vez de um JOIN? Qual das duas formas ficou mais clara?

---
### 🏁 Fechamento — Semana 10

**Nesta semana você aprendeu:**
- CRUD completo: INSERT, UPDATE e DELETE, sempre com WHERE
- Funções prontas de número, texto e data
- Subconsultas com IN e NOT IN
- Uma primeira olhada em variáveis/blocos anônimos, functions e transactions

**Próxima semana:** a Semana 11 conecta Python diretamente ao PostgreSQL para automatizar consultas e relatórios.

---
### Assinatura

Curso: **Análise de Dados com Python — SENAI (Turma T5)**
Semana 10 — SQL Avançado

*Prof. Especialista Cláudio F. Neves*